In [7]:
import pandas as pd
import os

In [11]:
new_df = pd.read_csv('combined.csv')

In [12]:
new_df.head()

,Unique ID,State/UT,District,Disease/Illness,No. of Cases,No. of Deaths,Date of Start of Outbreak,Date of Reporting,Current Status
0,AP/KRS/2024/01/01,Andhra Pradesh,Nandamuri Taraka Rama Rao (NTR),Acute Diarrheal Disease,30,0,30-12-2023,03-01-24,Under Control
1,AS/SBS/2024/01/02,Assam,Charaideo,Leptospirosis,1,1,02-01-24,02-01-24,Under Surveillance
2,AS/DAR/2024/01/03,Assam,Darrang,Chickenpox,6,0,30-12-2023,02-01-24,Under Surveillance
3,BH/JAH/2024/01/04,Bihar,Jehanabad,Chickenpox,8,0,06-01-24,06-01-24,Under Surveillance
4,GJ/AHM/2024/01/05,Gujarat,Ahmedabad,Acute Diarrheal Disease,16,0,01-12-2023,01-01-24,Under Surveillance


In [19]:
states=new_df['State/UT'].unique()
state_district_map = new_df.groupby("State/UT")["District"].unique().to_dict()

In [20]:
states

array(['Andhra Pradesh', 'Assam', 'Bihar', 'Gujarat', 'Karnataka',
       'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Tamil Nadu',
       'Uttarakhand', 'Chhattisgarh', 'Jharkhand', 'Uttar Pradesh',
       'West Bengal', 'Arunachal Pradesh', 'Puducherry',
       'D&N Haveli And Daman And Diu', 'Jammu and Kashmir', 'Punjab',
       'Meghalaya', 'Telangana', 'Haryana', 'Himachal Pradesh',
       'Rajasthan', 'Dadra And Nagar Haveli And Daman And Diu', 'Goa',
       'Manipur', 'Andaman & Nicobar Islands', 'Ladakh', 'State/UT',
       'Mizoram'], dtype=object)

In [21]:
state_district_map

{'Andaman & Nicobar Islands': array(['Nicobars', 'South Andamans'], dtype=object),
 'Andhra Pradesh': array(['Nandamuri Taraka Rama Rao (NTR)', 'Y.S.R.', 'Bapatla', 'Tirupati',
        'Palnadu', 'Sri Sathya Sai', 'Nandyal', 'Anantapur', 'Chittoor',
        'SPSR Nellore', 'Kurnool', 'NTR', 'Alluri Sitharama Raju',
        'Annamayya', 'Kakinada', 'Krishna', 'Prakasam', 'Srikakulam',
        'Anakapalli', 'Visakhapatanam', 'Vizianagaram',
        'Y. S. Rajasekhara Reddy',
        'Y.S.R. (Yeduguri Sandinti Rajasekhara Reddy)'], dtype=object),
 'Arunachal Pradesh': array(['East Kameng', 'Upper Subansiri', 'Papumpare', 'Longding',
        'Papum Pare', 'Tawang', 'Leparada', 'Lohit', 'Namsai',
        'West Kameng', 'West Siang'], dtype=object),
 'Assam': array(['Charaideo', 'Darrang', 'Hojai', 'Sonitpur', 'Lakhimpur',
        'Dhemaji', 'Bongaigaon', 'Goalpara', 'Udalguri', 'Jorhat',
        'Biswanath', 'Dhubri', 'Dibrugarh', 'Sivasagar', 'Kamrup',
        'Karbi Anglong', 'Kokrajhar',

In [31]:
state_district_pairs = new_df[["State/UT", "District", "Date of Start of Outbreak"]].drop_duplicates().sort_values("State/UT")
state_district_pairs.insert(0, "ID", range(1, len(state_district_pairs) + 1))
output_file = "state_district_combinations.csv"
state_district_pairs.to_csv(output_file, index=False)

## Normalizing Created at of trending data

In [34]:
import pandas as pd
from datetime import datetime

# Load CSV
updated_df = pd.read_csv('state_district_combinations.csv')

# Function to normalize date format
def normalize_date(date_str):
    for fmt in ("%d-%m-%Y", "%Y-%m-%d %H:%M:%S"):
        try:
            dt = datetime.strptime(date_str, fmt)
            return dt.strftime("%Y-%m-%d %H:%M:%S")  # Keep consistent format
        except ValueError:
            pass
    return None  # Return None if format is unrecognized

# Apply normalization
updated_df["created_at"] = updated_df["created_at"].astype(str).apply(normalize_date)

# Save back to CSV (overwrite the original file)
updated_df.to_csv('state_district_combinations.csv', index=False)

print("CSV file updated successfully!")


CSV file updated successfully!


## Disease data 

In [35]:
new_df = pd.read_csv('combined.csv')

In [39]:
state_district_pairs = new_df[["State/UT", "District", "Disease/Illness", "No. of Cases", "Date of Start of Outbreak"]].sort_values("State/UT")
state_district_pairs.insert(0, "ID", range(1, len(state_district_pairs) + 1))
output_file = "diseases.csv"
state_district_pairs.to_csv(output_file, index=False)

In [40]:
import pandas as pd
from datetime import datetime

# Load CSV
updated_df = pd.read_csv('diseases.csv')

# Function to normalize date format
def normalize_date(date_str):
    for fmt in ("%d-%m-%Y", "%Y-%m-%d %H:%M:%S"):
        try:
            dt = datetime.strptime(date_str, fmt)
            return dt.strftime("%Y-%m-%d %H:%M:%S")  # Keep consistent format
        except ValueError:
            pass
    return None  # Return None if format is unrecognized

# Apply normalization
updated_df["created_at"] = updated_df["created_at"].astype(str).apply(normalize_date)

# Save back to CSV (overwrite the original file)
updated_df.to_csv('diseases.csv', index=False)

print("CSV file updated successfully!")


CSV file updated successfully!


## Disease Data correction ID

In [109]:
disease_df= pd.read_csv('diseases.csv')
trending_df = pd.read_csv('state_district_combinations.csv')

In [115]:
for index, row in disease_df.iterrows():
    for _, t_row in trending_df.iterrows():
        if (
            row['state'] == t_row['state'] and
            row['city'] == t_row['city'] and
            row['created_at_dis'] == t_row['created_at_trend']
        ):
            disease_df.at[index, 'id'] = t_row['id']
            break  # Stop checking once a match is found

# Save the updated DataFrame if needed
disease_df.to_csv('updated_diseases.csv', index=True)

In [121]:
trending_data = pd.read_csv('state_district_combinations.csv')
trending_data['user'] = 3
trending_data.columns = ['id', 'state', 'city', 'created_at','user']
trending_data.to_csv('Webfinal/trending_data.csv',index=False)

In [120]:
disease_data = pd.read_csv('updated_diseases.csv')
disease_data = disease_data.drop(columns=['city'])
disease_data = disease_data.drop(columns=['state'])
disease_data = disease_data.drop(columns=['created_at_dis'])
disease_data.columns = ['id', 'trending_data', 'name', 'cases']
disease_data.to_csv('Webfinal/disease_data.csv',index=False)